In [1]:
import sys
sys.path.insert(0, '/home/jy/tza-pypsa')

# Now your imports will use the local version
import pypsa
from tz_pypsa.constraints import (
    constr_max_annual_utilisation_generator, 
    constr_min_annual_utilisation_generator,
    constr_max_annual_utilisation_links,
    constr_min_annual_utilisation_links,
    constr_max_annual_utilisation_storage_discharge, 
    constr_min_annual_utilisation_storage_discharge, 
    constr_max_annual_utilisation_storage_charge,     
    constr_min_annual_utilisation_storage_charge,     
    constr_soc_intraday_profile,
    constr_soc_weekly_profile,
    constr_production_target_max,
    constr_production_target_min,
    apply_ramping_cost
)

import plotly.express as px
import pandas as pd     
import numpy as np
import xarray as xr
import os
os.environ['GRB_LICENSE_FILE'] = '/home/jy/opt/gurobi/gurobi.lic'

In [2]:
n = pypsa.Network()
# n.import_from_netcdf("/home/jy/Backup/client-earth_CE-A-OCCTO-004_v3/platform_network.nc") # calibration
n.import_from_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-006_a-batteriessen/platform_network.solved.nc")

INFO:pypsa.io:Imported network platform_network.solved.nc has buses, carriers, generators, links, loads, storage_units


In [10]:
n.generators.to_csv('generator.csv')

In [13]:
price_diff_df = pd.read_csv("data/old/generators_price_differential.csv", index_col=0)
old_names = n.generators[(n.generators.type == "gas-unspecified") | (n.generators.type == "coal-unspecified") | (n.generators.type == "gas-hydrogen-cofiring") | (n.generators.type == "coal-ammonia-cofiring") | (n.generators.type == "gas-ccs")].index
n.mremove("Generator", old_names)
n.import_components_from_dataframe(price_diff_df, "Generator")

In [4]:
# n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="coal").columns] = n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="coal-unspecified").columns].max().max()
# n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="gas").columns] = n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="gas-unspecified").columns].max().max()
n.generators_t.marginal_cost.loc[:, n.generators_t.marginal_cost.filter(like="nuclear").columns] = 12

In [48]:
n_solved = pypsa.Network()
n_solved.import_from_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-005_v3-dispatch-sen-nuc-testing-rampingcost/platform_network.solved.nc")

INFO:pypsa.io:Imported network platform_network.solved.nc has buses, carriers, generators, links, loads, storage_units


In [51]:
n_solved.generators.to_csv("nuclear_asset_level_data.csv")

In [45]:
n_solved.generators.loc[n_solved.generators.was_extendable][['p_nom', 'p_nom_expansion_opt']]

,p_nom,p_nom_expansion_opt
Generator,,
wind-offshore-unspecified:GRIDREGION-JPN-SH,1982.0,1981.999290
wind-offshore-unspecified:GRIDREGION-JPN-HR,1300.0,1300.000000
wind-offshore-unspecified:GRIDREGION-JPN-CB,10378.0,10377.990947
wind-offshore-unspecified:GRIDREGION-JPN-HK,14651.0,14650.000005
wind-offshore-unspecified:GRIDREGION-JPN-KA,1150.0,1149.999901
...,...,...
gas-hydrogen-cofiring:GRIDREGION-JPN-KA,1448.0,1448.000000
gas-hydrogen-cofiring:GRIDREGION-JPN-CG,1056.0,1056.000000
gas-hydrogen-cofiring:GRIDREGION-JPN-TK,3779.0,3779.000000


In [ ]:
n.generators.loc[n_solved.generators.p_nom_extendable, "p_nom"] = np.ceil(n_solved.generators.loc[n_solved.generators.p_nom_extendable, "p_nom_opt"])
n.storage_units.loc[n_solved.storage_units.p_nom_extendable, "p_nom"] = np.ceil(n_solved.storage_units.loc[n_solved.storage_units.p_nom_extendable, "p_nom_opt"])
# n.storage_units.loc[n_solved.storage_units.type == 'utility-scale', "p_nom"] = np.ceil(n_solved.storage_units.loc[n_solved.storage_units.type == 'utility-scale', "p_nom"])

In [ ]:
n.generators.p_nom_extendable = False
n.storage_units.p_nom_extendable = False

In [ ]:
new_nuclear_df = pd.read_csv("nuclear_asset_level_data.csv", index_col=0)
old_nuclear_names = n.generators[n.generators.type == "nuclear"].index
n.mremove("Generator", old_nuclear_names)
n.import_components_from_dataframe(new_nuclear_df, "Generator")

In [ ]:
n_nuc = pypsa.Network()
n_nuc.import_from_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-005_v1/platform_network.nc")

In [ ]:
n.generators.loc[n.generators.type == 'nuclear', 'p_nom'] = n_nuc.generators.loc[n_nuc.generators.type == 'nuclear', 'p_nom']

In [ ]:
n.generators.loc[:, "p_nom"] = np.ceil(n_solved.generators.loc[:, "p_nom_opt"])
n.storage_units.loc[:, "p_nom"] = np.ceil(n_solved.storage_units.loc[:, "p_nom_opt"])

In [3]:
n.generators['carrier'] = n.generators['type']
n.links['carrier'] = n.links['type']
n.storage_units['carrier'] = n.storage_units['type']

In [4]:
all_carriers = (
    n.generators.carrier.unique().tolist()
    + n.storage_units.carrier.unique().tolist()
    + n.links.carrier.unique().tolist()
)
missing_carriers = set(all_carriers) - set(n.carriers.index)
if missing_carriers:
    n.add("Carrier", missing_carriers)

n.generators.build_year = 2040
n.storage_units.build_year = 2040
n.links.build_year = 2040

In [ ]:
# n.storage_units_t.state_of_charge_set[:] = float("nan")
# n.storage_units_t.state_of_charge_set[n.storage_units_t.state_of_charge_set.notna().any(axis=1)]

In [ ]:
# p_nom for batteries
n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'p_nom'] = 610
n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'p_nom'] = 1711
n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'p_nom'] = 6549
n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'p_nom'] = 3034
n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'p_nom'] = 616
n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'p_nom'] = 3327
n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'p_nom'] = 1273
n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'p_nom'] = 592
n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'p_nom'] = 1859

In [5]:
n.generators.loc[n.generators.carrier == 'wind-offshore-unspecified', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'wind-onshore', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'photovoltaic-unspecified', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'geothermal-unspecified', 'p_nom_extendable'] = True

n.generators.loc[n.generators.carrier == 'gas-ccs', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'coal-ammonia-cofiring', 'p_nom_extendable'] = True
n.generators.loc[n.generators.carrier == 'gas-hydrogen-cofiring', 'p_nom_extendable'] = True
n.storage_units.loc[n.storage_units.carrier == 'utility-scale', 'p_nom_extendable'] = True

In [ ]:
# p_nom for nuclear across different scenario
# Scenario A - 20% nuclear generation share which is the default capacity configuration on DWH

# # Scenario B - 12% nuclear generation share
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom'] = 2070
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom'] = 825
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom'] = 2712
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom'] = 0
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom'] = 1206
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom'] = 4100
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom'] = 820
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom'] = 890
# n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom'] = 4140

# Scenario C - 16% nuclear generation share
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom'] = 2070
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom'] = 2208
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom'] = 3812
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom'] = 0
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom'] = 1206
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom'] = 6578
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom'] = 2193
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom'] = 890
n.generators.loc[(n.generators.carrier == 'nuclear') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom'] = 4140

In [6]:
# set p_nom_min = p_nom to avoid capacity retirement
n.generators.p_nom_min = n.generators.p_nom

In [7]:
# Renewable p_nom_max
# p_nom_max for geothermal
n.generators.loc[n.generators.carrier == 'geothermal-unspecified', 'p_nom_max'] = n.generators.loc[n.generators.carrier == 'geothermal-unspecified', 'p_nom'] * 1.5

# p_nom_max for solar
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 8305 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 33780 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 60231 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 38999 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 4970 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 23097 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 26285 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 13496 
n.generators.loc[(n.generators.carrier == 'photovoltaic-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 49404

# p_nom_max for onshore wind
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 6290
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 18246
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 3890
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 1221
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 1789
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 2404
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 2144
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 1950
n.generators.loc[(n.generators.carrier == 'wind-onshore') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 3039

# p_nom_max for offshore wind
# Taking highest quality sites for each region
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 45106
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 24536
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 16176
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 10378
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 1300
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 1150
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 843
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 1982 
n.generators.loc[(n.generators.carrier == 'wind-offshore-unspecified') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 13141

# # p_nom_max for batteries
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 610
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 1711
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 6549
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 3034
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 616
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 3327
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 1273
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 592
# n.storage_units.loc[(n.storage_units.carrier == 'utility-scale') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 1859

In [8]:
# p_nom_max for coal-ammonia-cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 700
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 405
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 997
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 1000
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 257
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 1400
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 810
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 102 
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 372

In [9]:
# p_nom_max for gas-hydrogen-cofiring
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_nom_max'] = 437
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_nom_max'] = 352
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_nom_max'] = 3779
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_nom_max'] = 1306
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_nom_max'] = 184
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_nom_max'] = 1448
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_nom_max'] = 1056
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_nom_max'] = 67
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_nom_max'] = 316

In [10]:
n.generators.groupby('type')[['p_nom', 'p_nom_max']].sum()

,p_nom,p_nom_max
type,,
biomass,7220.0,inf
coal-ammonia-cofiring,3100.0,6043.0
coal-unspecified,20997.0,inf
gas-ccs,250.0,inf
gas-hydrogen-cofiring,1200.0,8945.0
gas-unspecified,20133.0,inf
geothermal-unspecified,1550.0,2325.0
hydro-reservoir-and-run-of-river,23360.0,inf
nuclear,27814.0,inf


In [14]:
n.generators.marginal_cost[n.generators.carrier == 'coal-ammonia-cofiring']

Generator
coal-ammonia-cofiring:GRIDREGION-JPN-SH    103.66
coal-ammonia-cofiring:GRIDREGION-JPN-HR    100.10
coal-ammonia-cofiring:GRIDREGION-JPN-CB    100.79
coal-ammonia-cofiring:GRIDREGION-JPN-HK    106.04
coal-ammonia-cofiring:GRIDREGION-JPN-KA    100.67
coal-ammonia-cofiring:GRIDREGION-JPN-CG    103.66
coal-ammonia-cofiring:GRIDREGION-JPN-TK    100.61
coal-ammonia-cofiring:GRIDREGION-JPN-TH    103.24
coal-ammonia-cofiring:GRIDREGION-JPN-KY    106.27
Name: marginal_cost, dtype: float64

In [ ]:
n.storage_units

In [15]:
n.generators.loc[(n.generators.carrier == 'coal-subcritical'), 'ramp_limit_up'] = 0.4
n.generators.loc[(n.generators.carrier == 'coal-subcritical'), 'ramp_limit_down'] = 0.4

n.generators.loc[(n.generators.carrier == 'coal-supercritical'), 'ramp_limit_up'] = 0.4
n.generators.loc[(n.generators.carrier == 'coal-supercritical'), 'ramp_limit_down'] = 0.4

n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring'), 'ramp_limit_up'] = 0.4
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring'), 'ramp_limit_down'] = 0.4

n.generators.loc[(n.generators.carrier == 'gas-conventional'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-conventional'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'gas-combined-cycle'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'gas-ccs'), 'ramp_limit_up'] = 0.9
n.generators.loc[(n.generators.carrier == 'gas-ccs'), 'ramp_limit_down'] = 0.9

n.generators.loc[(n.generators.carrier == 'nuclear'), 'ramp_limit_up'] = 0.6
n.generators.loc[(n.generators.carrier == 'nuclear'), 'ramp_limit_down'] = 0.6

In [16]:
n.storage_units.loc[n.storage_units.carrier == 'utility-scale', 'efficiency_store'] = 0.92
n.storage_units.loc[n.storage_units.carrier == 'utility-scale', 'efficiency_dispatch'] = 0.92
n.storage_units.loc[n.storage_units.carrier == 'utility-scale', 'marginal_cost'] = 1
n.storage_units.loc[n.storage_units.carrier == 'hydro-pumped-storage-unspecified', 'marginal_cost'] = 1

In [ ]:
n.generators.loc[n.generators.carrier == 'wind-offshore-unspecified', 'marginal_cost'] = -1
n.generators.loc[n.generators.carrier == 'wind-onshore', 'marginal_cost'] = -1
n.generators.loc[n.generators.carrier == 'photovoltaic-unspecified', 'marginal_cost'] = -1

In [17]:
n.generators.loc[n.generators.carrier == 'nuclear', 'p_min_pu'] = 0.7
n.generators.loc[n.generators.carrier == 'nuclear', 'p_max_pu'] = 0.7

In [ ]:
n.generators.loc[n.generators.carrier == 'nuclear', 'max_ramps_per_day'] = 2

In [18]:
n.generators_t.p_min_pu = n.generators_t.p_max_pu.filter(regex='biomass|geothermal|hydro')

In [19]:
# p_max_pu - coal-subcritical
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_max_pu'] = 0.46
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.57
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.62
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.63
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.59
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.51

# p_max_pu - coal-supercritical
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_max_pu'] = 0.46
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.57
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.62
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.63
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.59
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.51

# p_max_pu - coal-ammonia cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'p_max_pu'] = 0.46
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'p_max_pu'] = 0.57
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'p_max_pu'] = 0.62
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'p_max_pu'] = 0.63
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'p_max_pu'] = 0.60
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'p_max_pu'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'p_max_pu'] = 0.59
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'p_max_pu'] = 0.51

In [20]:
# max_utilisation_rate
# coal - subcritical
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.22
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.29
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.36
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.27
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.30
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.24

# coal - supercritical
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.45
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.58
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.68
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.72
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.67
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.68
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.54
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.59
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.48

# coal-ammonia cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.55
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.51
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.41
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.45
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.365

# gas-conventional
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

# gas-combined-cycle
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

# gas-advanced-combined-cycle
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

# gas-more-advanced-combined-cycle
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

# gas-ccs
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

# gas-hydrogen-cofiring
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.70
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.70

# # nuclear
# n.generators.loc[(n.generators.carrier == 'nuclear'), 'max_utilisation_rate'] = 0.70

In [21]:
# min_utilisation_rate
# coal - subcritical
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.22
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.29
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-subcritical') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.24

# coal - supercritical
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-supercritical') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.365

# coal-ammonia-cofiring
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.34
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.445
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.52
n.generators.loc[(n.generators.carrier == 'coal-ammonia-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.365


# gas - calibration constraints
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-conventional') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

# gas - calibration constraints
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

# gas - calibration constraints
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

# gas - calibration constraints
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-more-advanced-combined-cycle') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

# Assume gas-hydrogen-cofiring to follow the same min utilisation rate as gas to reflect the same level of operational constraints
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-hydrogen-cofiring') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

# Assume gas-ccs to follow the same min utilisation rate as gas to reflect the same level of operational constraints
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HK'), 'min_utilisation_rate'] = 0.14
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.095
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.16
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.23
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.10
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.225
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.05
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-SH'), 'min_utilisation_rate'] = 0.085
n.generators.loc[(n.generators.carrier == 'gas-ccs') & (n.generators.bus == 'GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.04

In [22]:
# max_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.48
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.5875
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.44
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.02
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.04
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.33
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.18
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.09
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.19
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.40
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.00
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.34
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.83

In [23]:
# max_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.48 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-HK'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.5875 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-TH'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.44 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-TK'), 'max_utilisation_rate'] = 0.25 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.02 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.04 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.33 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.18 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'max_utilisation_rate'] = 0.09 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CB'), 'max_utilisation_rate'] = 0.25 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.19 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KA'), 'max_utilisation_rate'] = 0.40 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KY'), 'max_utilisation_rate'] = 0.00 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-CG'), 'max_utilisation_rate'] = 0.34 + 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KY~GRIDREGION-JPN-SH'), 'max_utilisation_rate'] = 0.83 + 0.025

In [24]:
# min_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.48
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.5875
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.44
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.04
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.32
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.25
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.09
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.19
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.40
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.19
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.98
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.91

In [25]:
# min_utilisation_rate - transmission
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HK~GRIDREGION-JPN-TH'), 'min_utilisation_rate'] = 0.48 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TH~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.5875 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-TK~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.44 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-TK'), 'min_utilisation_rate'] = 0.25 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CB~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.04 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-HR~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.32 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CB'), 'min_utilisation_rate'] = 0.25 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-HR'), 'min_utilisation_rate'] = 0.09 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-KA~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.19 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.40 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-CG~GRIDREGION-JPN-KY'), 'min_utilisation_rate'] = 0.19 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-KA'), 'min_utilisation_rate'] = 0.98 - 0.025
n.links.loc[(n.links.index == 'transmission:GRIDREGION-JPN-SH~GRIDREGION-JPN-CG'), 'min_utilisation_rate'] = 0.91 - 0.025

In [26]:
# max_utilisation_rate
# hydro-pumped-storage
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'discharge_min_utilisation_rate'] = 0.126
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HK'), 'charge_max_utilisation_rate'] = 0.180

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'discharge_min_utilisation_rate'] = 0.136
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TH'), 'charge_max_utilisation_rate'] = 0.195

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'discharge_min_utilisation_rate'] = 0.121
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-TK'), 'charge_max_utilisation_rate'] = 0.174

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'discharge_min_utilisation_rate'] = 0.059
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CB'), 'charge_max_utilisation_rate'] = 0.085

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'discharge_min_utilisation_rate'] = 0.055
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-HR'), 'charge_max_utilisation_rate'] = 0.080

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'discharge_min_utilisation_rate'] = 0.056
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KA'), 'charge_max_utilisation_rate'] = 0.080

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'discharge_min_utilisation_rate'] = 0.056
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-CG'), 'charge_max_utilisation_rate'] = 0.081

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'discharge_min_utilisation_rate'] = 0.064
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-SH'), 'charge_max_utilisation_rate'] = 0.092

n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'discharge_min_utilisation_rate'] = 0.057
n.storage_units.loc[(n.storage_units.carrier == 'hydro-pumped-storage-unspecified') & (n.storage_units.bus == 'GRIDREGION-JPN-KY'), 'charge_max_utilisation_rate'] = 0.083

In [ ]:
n.generators.to_csv("generators.csv")
n.links.to_csv("links.csv")
n.storage_units.to_csv("storage.csv")

In [ ]:
ramping_costs = {
    'coal-subcritical': 299,
    'coal-supercritical': 264,
    'coal-ammonia-cofiring': 300,
    'gas-conventional': 365,
    'gas-combined-cycle': 209,
    'gas-advanced-combined-cycle': 178,
    'gas-more-advanced-combined-cycle': 173,
    'gas-hydrogen-cofiring': 173,
    'gas-ccs': 173
}

In [27]:
n.optimize.create_model()
# constr_max_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified') # Calibration
# constr_min_annual_utilisation_generator(n, carriers='coal-unspecified|gas-unspecified') # Calibration
constr_max_annual_utilisation_generator(n, carriers='coal|gas') # Set max annual utilisation for these generators
constr_min_annual_utilisation_generator(n, carriers='coal|gas') # Set min annual utilisation for these generators
constr_max_annual_utilisation_links(n, carriers='transmission') # Set max annual utilisation for these links
constr_min_annual_utilisation_links(n, carriers='transmission') # Set min annual utilisation for these links
constr_min_annual_utilisation_storage_discharge(n, carriers='hydro-pumped-storage-unspecified') # Set min annual utilisation for these storage units
constr_max_annual_utilisation_storage_charge(n, carriers='hydro-pumped-storage-unspecified') # Set min annual utilisation for these storage units
constr_soc_intraday_profile(
    n, 
    max_csv="/home/jy/tza-pypsa/ClientEarth/data/storage_profiles/state_of_charge_intraday_profile_annual_max.csv",
    min_csv="/home/jy/tza-pypsa/ClientEarth/data/storage_profiles/state_of_charge_intraday_profile_annual_min.csv"
)
constr_soc_weekly_profile(
    n, 
    max_csv="/home/jy/tza-pypsa/ClientEarth/data/storage_profiles/state_of_charge_weekly_profile_annual_max.csv",
    min_csv="/home/jy/tza-pypsa/ClientEarth/data/storage_profiles/state_of_charge_weekly_profile_annual_min.csv",
    day_shift=2
)
constr_production_target_min(n, 
                            ['GRIDREGION-JPN-SH', 'GRIDREGION-JPN-HR', 'GRIDREGION-JPN-CB', 
                            'GRIDREGION-JPN-HK', 'GRIDREGION-JPN-KA', 'GRIDREGION-JPN-CG', 
                            'GRIDREGION-JPN-TK', 'GRIDREGION-JPN-TH', 'GRIDREGION-JPN-KY'],
                            ['wind-offshore-unspecified','photovoltaic-unspecified', 'wind-onshore', 'geothermal-unspecified', 'biomass', 'hydro-reservoir-and-run-of-river'],
                            0.50)

constr_production_target_max(n, 
                            ['GRIDREGION-JPN-SH', 'GRIDREGION-JPN-HR', 'GRIDREGION-JPN-CB', 
                            'GRIDREGION-JPN-HK', 'GRIDREGION-JPN-KA', 'GRIDREGION-JPN-CG', 
                            'GRIDREGION-JPN-TK', 'GRIDREGION-JPN-TH', 'GRIDREGION-JPN-KY'],
                            ['wind-offshore-unspecified','photovoltaic-unspecified', 'wind-onshore', 'geothermal-unspecified', 'biomass', 'hydro-reservoir-and-run-of-river'],
                            0.505)

# apply_ramping_cost(n, ramping_costs)


Index(['coal-subcritical:GRIDREGION-JPN-HK',
       'coal-subcritical:GRIDREGION-JPN-TH',
       'coal-subcritical:GRIDREGION-JPN-TK',
       'coal-subcritical:GRIDREGION-JPN-HR',
       'coal-subcritical:GRIDREGION-JPN-KA',
       'coal-subcritical:GRIDREGION-JPN-SH',
       'coal-subcritical:GRIDREGION-JPN-CG',
       'coal-subcritical:GRIDREGION-JPN-KY',
       'coal-supercritical:GRIDREGION-JPN-HK',
       'coal-supercritical:GRIDREGION-JPN-TH',
       'coal-supercritical:GRIDREGION-JPN-TK',
       'coal-supercritical:GRIDREGION-JPN-CB',
       'coal-supercritical:GRIDREGION-JPN-HR',
       'coal-supercritical:GRIDREGION-JPN-KA',
       'coal-supercritical:GRIDREGION-JPN-SH',
       'coal-supercritical:GRIDREGION-JPN-CG',
       'coal-supercritical:GRIDREGION-JPN-KY',
       'gas-advanced-combined-cycle:GRIDREGION-JPN-TH',
       'gas-advanced-combined-cycle:GRIDREGION-JPN-TK',
       'gas-advanced-combined-cycle:GRIDREGION-JPN-CB',
       'gas-advanced-combined-cycle:GRIDREGION-JP

['gas-ccs:GRIDREGION-JPN-SH', 'gas-ccs:GRIDREGION-JPN-HR', 'gas-ccs:GRIDREGION-JPN-CB', 'gas-ccs:GRIDREGION-JPN-HK', 'gas-ccs:GRIDREGION-JPN-KA', 'gas-ccs:GRIDREGION-JPN-CG', 'gas-ccs:GRIDREGION-JPN-TK', 'gas-ccs:GRIDREGION-JPN-TH', 'gas-ccs:GRIDREGION-JPN-KY', 'coal-ammonia-cofiring:GRIDREGION-JPN-SH', 'coal-ammonia-cofiring:GRIDREGION-JPN-HR', 'coal-ammonia-cofiring:GRIDREGION-JPN-CB', 'coal-ammonia-cofiring:GRIDREGION-JPN-HK', 'coal-ammonia-cofiring:GRIDREGION-JPN-KA', 'coal-ammonia-cofiring:GRIDREGION-JPN-CG', 'coal-ammonia-cofiring:GRIDREGION-JPN-TK', 'coal-ammonia-cofiring:GRIDREGION-JPN-TH', 'coal-ammonia-cofiring:GRIDREGION-JPN-KY', 'gas-hydrogen-cofiring:GRIDREGION-JPN-SH', 'gas-hydrogen-cofiring:GRIDREGION-JPN-HR', 'gas-hydrogen-cofiring:GRIDREGION-JPN-CB', 'gas-hydrogen-cofiring:GRIDREGION-JPN-HK', 'gas-hydrogen-cofiring:GRIDREGION-JPN-KA', 'gas-hydrogen-cofiring:GRIDREGION-JPN-CG', 'gas-hydrogen-cofiring:GRIDREGION-JPN-TK', 'gas-hydrogen-cofiring:GRIDREGION-JPN-TH', 'gas-hy

In [28]:
n.optimize.solve_model(
    solver_name='gurobi',
    solver_options={
        'threads': 8,
        'method': 2, # barrier
        'crossover': 0,
        'BarConvTol': 1.e-6,
        'Seed': 123,
        'AggFill': 0,
        'PreDual': 0,
        'LogFile': 'gurobi.log',
        'LogToConsole': 1  
    },
    io_api="direct",
    env=None,
)

INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.model:Solver options:
 - threads: 8
 - method: 2
 - crossover: 0
 - BarConvTol: 1e-06
 - Seed: 123
 - AggFill: 0
 - PreDual: 0
 - LogFile: gurobi.log
 - LogToConsole: 1


Set parameter WLSAccessID


INFO:gurobipy:Set parameter WLSAccessID


Set parameter WLSSecret


INFO:gurobipy:Set parameter WLSSecret


...

Optimal objective 6.89090120e+10


INFO:gurobipy:Optimal objective 6.89090120e+10


INFO:gurobipy:


INFO:gurobipy:Warning: environment still referenced so free is deferred (Continue to use WLS)
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 1857166 primals, 5388059 duals
Objective: 6.89e+10
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Generator-fix-p-ramp_limit_up, Generator-fix-p-ramp_limit_down, Link-fix-p-lower, Link-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance were n

('ok', 'optimal')

In [30]:
n.optimize.fix_optimal_capacities()

In [31]:
n.generators

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_min_pu,p_max_pu,...,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt,emission_activity_ratio,opex_fixed,max_utilisation_rate,min_utilisation_rate
Generator,,,,,,,,,,,,,,,,,,,,,
wind-offshore-unspecified:GRIDREGION-JPN-SH,GRIDREGION-JPN-SH,PQ,wind-offshore-unspecified,1981.999988,0.0,False,1700.0,1982.0,0.0,1.0,...,NaN,NaN,1.0,1.0,1.0,1981.999988,NaN,65507.74854,NaN,NaN
wind-offshore-unspecified:GRIDREGION-JPN-HR,GRIDREGION-JPN-HR,PQ,wind-offshore-unspecified,1300.000000,0.0,False,1300.0,1300.0,0.0,1.0,...,NaN,NaN,1.0,1.0,1.0,1300.000000,NaN,65507.74854,NaN,NaN
wind-offshore-unspecified:GRIDREGION-JPN-CB,GRIDREGION-JPN-CB,PQ,wind-offshore-unspecified,10377.999893,0.0,False,1350.0,10378.0,0.0,1.0,...,NaN,NaN,1.0,1.0,1.0,10377.999893,NaN,65507.74854,NaN,NaN
wind-offshore-unspecified:GRIDREGION-JPN-HK,GRIDREGION-JPN-HK,PQ,wind-offshore-unspecified,14650.000001,0.0,False,14650.0,45106.0,0.0,1.0,...,NaN,NaN,1.0,1.0,1.0,14650.000001,NaN,65507.74854,NaN,NaN
wind-offshore-unspecified:GRIDREGION-JPN-KA,GRIDREGION-JPN-KA,PQ,wind-offshore-unspecified,1149.999981,0.0,False,900.0,1150.0,0.0,1.0,...,NaN,NaN,1.0,1.0,1.0,1149.999981,NaN,65507.74854,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
gas-more-advanced-combined-cycle:GRIDREGION-JPN-CB,GRIDREGION-JPN-CB,PQ,gas-more-advanced-combined-cycle,2625.000000,0.0,False,0.0,inf,0.0,1.0,...,0.9,0.9,1.0,1.0,1.0,2625.000000,0.198,79141.06703,0.7,0.230
gas-more-advanced-combined-cycle:GRIDREGION-JPN-HR,GRIDREGION-JPN-HR,PQ,gas-more-advanced-combined-cycle,159.000000,0.0,False,0.0,inf,0.0,1.0,...,0.9,0.9,1.0,1.0,1.0,159.000000,0.198,79141.06703,0.7,0.100
gas-more-advanced-combined-cycle:GRIDREGION-JPN-KA,GRIDREGION-JPN-KA,PQ,gas-more-advanced-combined-cycle,2092.000000,0.0,False,0.0,inf,0.0,1.0,...,0.9,0.9,1.0,1.0,1.0,2092.000000,0.198,79141.06703,0.7,0.225


In [29]:
n.generators

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_min_pu,p_max_pu,...,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt,emission_activity_ratio,opex_fixed,max_utilisation_rate,min_utilisation_rate
Generator,,,,,,,,,,,,,,,,,,,,,
wind-offshore-unspecified:GRIDREGION-JPN-SH,GRIDREGION-JPN-SH,PQ,wind-offshore-unspecified,1700.0,0.0,True,1700.0,1982.0,0.0,1.0,...,NaN,NaN,1.0,1.0,1.0,1981.999988,NaN,65507.74854,NaN,NaN
wind-offshore-unspecified:GRIDREGION-JPN-HR,GRIDREGION-JPN-HR,PQ,wind-offshore-unspecified,1300.0,0.0,True,1300.0,1300.0,0.0,1.0,...,NaN,NaN,1.0,1.0,1.0,1300.000000,NaN,65507.74854,NaN,NaN
wind-offshore-unspecified:GRIDREGION-JPN-CB,GRIDREGION-JPN-CB,PQ,wind-offshore-unspecified,1350.0,0.0,True,1350.0,10378.0,0.0,1.0,...,NaN,NaN,1.0,1.0,1.0,10377.999893,NaN,65507.74854,NaN,NaN
wind-offshore-unspecified:GRIDREGION-JPN-HK,GRIDREGION-JPN-HK,PQ,wind-offshore-unspecified,14650.0,0.0,True,14650.0,45106.0,0.0,1.0,...,NaN,NaN,1.0,1.0,1.0,14650.000001,NaN,65507.74854,NaN,NaN
wind-offshore-unspecified:GRIDREGION-JPN-KA,GRIDREGION-JPN-KA,PQ,wind-offshore-unspecified,900.0,0.0,True,900.0,1150.0,0.0,1.0,...,NaN,NaN,1.0,1.0,1.0,1149.999981,NaN,65507.74854,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
gas-more-advanced-combined-cycle:GRIDREGION-JPN-CB,GRIDREGION-JPN-CB,PQ,gas-more-advanced-combined-cycle,2625.0,0.0,False,0.0,inf,0.0,1.0,...,0.9,0.9,1.0,1.0,1.0,2625.000000,0.198,79141.06703,0.7,0.230
gas-more-advanced-combined-cycle:GRIDREGION-JPN-HR,GRIDREGION-JPN-HR,PQ,gas-more-advanced-combined-cycle,159.0,0.0,False,0.0,inf,0.0,1.0,...,0.9,0.9,1.0,1.0,1.0,159.000000,0.198,79141.06703,0.7,0.100
gas-more-advanced-combined-cycle:GRIDREGION-JPN-KA,GRIDREGION-JPN-KA,PQ,gas-more-advanced-combined-cycle,2092.0,0.0,False,0.0,inf,0.0,1.0,...,0.9,0.9,1.0,1.0,1.0,2092.000000,0.198,79141.06703,0.7,0.225


In [ ]:
n.buses_t.marginal_price.describe()

In [ ]:
n.statistics()

In [ ]:
n.export_to_netcdf("/home/jy/tz-amp/.amp/client-earth_CE-A-SEP7-PoC_alpha-9/platform_network.solved.nc")